In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import SegformerForSemanticSegmentation
from tqdm import tqdm
from pathlib import Path

In [2]:
class SegFormerDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(512, 512)):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        
        self.mask_filenames = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        self.image_path_map = {}
        for path in Path(image_dir).rglob("*.jpg"):
            self.image_path_map[path.name] = str(path)
            

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_name = self.mask_filenames[idx]
        img_name = mask_name.replace('.png', '.jpg')

        img_path = self.image_path_map.get(img_name)
        
        if img_path is None:
            raise FileNotFoundError(f"'{img_name}' 파일을 찾을 수 없습니다.")

        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") 

        image = image.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)

        img_tensor = transforms.ToTensor()(image)
        img_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img_tensor)

        mask_tensor = torch.tensor(np.array(mask), dtype=torch.long)

        return img_tensor, mask_tensor

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

IMAGE_DIR = r"D:\Study\HumanStudy\Dataset\Training\01.원천데이터"
MASK_DIR = r"D:\Study\HumanStudy\Dataset\Training_Masks"

train_dataset = SegFormerDataset(IMAGE_DIR, MASK_DIR, img_size=(512, 512))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=11,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00006)

cuda


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
num_epochs = 5
best_loss = float('inf')

print(f"\n 총 학습 데이터 수: {len(train_dataset):,}장")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        
        outputs = model(pixel_values=images, labels=masks)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_loss = running_loss / len(train_loader)
    print(f" Epoch {epoch+1} 평균 Loss: {epoch_loss:.4f}")
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), 'best_segformer_b0_crack.pth')
        print(f"  weight 저장 (best_segformer_b0_crack.pth)")


 총 학습 데이터 수: 98,398장


Epoch 1/5: 100%|██████████████████████████████████████████████████| 12300/12300 [2:12:13<00:00,  1.55it/s, loss=0.1892]


 Epoch 1 평균 Loss: 0.2162
  weight 저장 (best_segformer_b0_crack.pth)


Epoch 2/5: 100%|██████████████████████████████████████████████████| 12300/12300 [2:09:36<00:00,  1.58it/s, loss=0.3621]


 Epoch 2 평균 Loss: 0.1472
  weight 저장 (best_segformer_b0_crack.pth)


...

 Epoch 3 평균 Loss: 0.1341
  weight 저장 (best_segformer_b0_crack.pth)


Epoch 4/5: 100%|██████████████████████████████████████████████████| 12300/12300 [2:08:43<00:00,  1.59it/s, loss=0.0976]


 Epoch 4 평균 Loss: 0.1258
  weight 저장 (best_segformer_b0_crack.pth)


Epoch 5/5: 100%|██████████████████████████████████████████████████| 12300/12300 [2:08:15<00:00,  1.60it/s, loss=0.1674]

 Epoch 5 평균 Loss: 0.1201
  weight 저장 (best_segformer_b0_crack.pth)
